In [1]:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from dotenv import load_dotenv

e:\Langgraph CampusX\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
# define the hugging face endpoint
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=2048,
)

# define the model
model = ChatHuggingFace(llm=llm)

In [4]:
def chat_node(state: MessagesState):

    messages = trim_messages(
        messages=state["messages"],
        max_tokens=180,
        token_counter=count_tokens_approximately,
        strategy="last"
    )

    print("Current Token Count:", count_tokens_approximately(messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages" : [response]}

In [5]:
graph = StateGraph(MessagesState)
graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")

In [6]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config = {"configurable": {"thread_id": "chat-1"}}

result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Nitish."}]},
    config,
)

result["messages"][-1].content

Current Token Count: 10
Hi, my name is Nitish.


"Hello Nitish! Nice to meet you. How can I assist you today? Is there any specific topic or question you'd like to discuss?"

In [9]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "I am learning Artificial Neural Networks in Deep Learning."}]},
    config,
)

result["messages"][-1].content

Current Token Count: 65
Hi, my name is Nitish.
Hello Nitish! Nice to meet you. How can I assist you today? Is there any specific topic or question you'd like to discuss?
I am learning Artificial Neural Networks in Deep Learning.


'Great! Learning about Artificial Neural Networks (ANNs) is a fantastic start in the field of deep learning. ANNs are a fundamental building block in deep learning and are inspired by the way the human brain functions. Here’s a brief overview to get you started:\n\n### What Are Artificial Neural Networks?\nArtificial Neural Networks are computational models that are designed to mimic the structure and functioning of biological neural networks. They are used for a wide range of applications, including image and speech recognition, natural language processing, and more.\n\n### Key Components of ANNs\n1. **Neurons (Nodes):** The basic units of computation.\n2. **Layers:** ANNs typically consist of multiple layers: an input layer, one or more hidden layers, and an output layer.\n3. **Connections:** Each neuron is connected to the neurons in the next layer. Each connection has a weight.\n4. **Activation Function:** A function that determines the output of a neuron based on its input. Common

In [10]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory concept in RNN/ LSTM?"}]},
    config,
)

result["messages"][-1].content

Current Token Count: 18
Can you explain short term memory concept in RNN/ LSTM?


'Certainly! Short-term memory is a critical concept when discussing Recurrent Neural Networks (RNNs) and their variant, Long Short-Term Memory networks (LSTMs). Both types of networks are designed to process sequential data, such as time-series data or sentences, where the order of the elements is important.\n\n### Recurrent Neural Networks (RNNs)\n\nIn an RNN, the model has an internal state that is updated with each input. This state is essentially a form of short-term memory, as it captures information from previous inputs in the sequence and uses it to inform the current output. However, RNNs suffer from the vanishing gradient problem, which can make it difficult for the network to remember information from very far back in the sequence (i.e., long-term dependencies).\n\n### Long Short-Term Memory (LSTM) Networks\n\nLSTM networks are designed to overcome the limitations of RNNs by incorporating a more sophisticated internal state mechanism. In an LSTM, the memory is organized into 

In [11]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count: 8
What is my name?


"To answer your question, I would need more context. I don't have access to personal information about individuals by default. If you're asking in the context of a specific chat session or profile, your name would be listed there. If you tell me your name, I can use it to address you personally, but I won't be able to know it without you telling me. Can you please tell me your name so I can assist you better?"

In [12]:
for item in workflow.get_state({"configurable": {"thread_id": "chat-1"}}).values['messages']:
    print(item.content)
    print('-'*120)

Hi, my name is Nitish.
------------------------------------------------------------------------------------------------------------------------
Hello Nitish! Nice to meet you. How can I assist you today? Is there any specific topic or question you'd like to discuss?
------------------------------------------------------------------------------------------------------------------------
I am learning Artificial Neural Networks in Deep Learning.
------------------------------------------------------------------------------------------------------------------------
Great! Learning about Artificial Neural Networks (ANNs) is a fantastic start in the field of deep learning. ANNs are a fundamental building block in deep learning and are inspired by the way the human brain functions. Here’s a brief overview to get you started:

### What Are Artificial Neural Networks?
Artificial Neural Networks are computational models that are designed to mimic the structure and functioning of biological neura